In [ ]:
# Old version code: ploidy can only change between 44~46

# This code is used to isimulate the unstable ploidy copy number during amitosis.  T.thermophila  have  an  unknown  mechanism   
# that controls macronuclear chromosome copy number during asexual division. Thus here we investigate the effects of unstable
# ploidy copy number during amitosis. In this code, the ploidy only changes during amitosis, and the sexual reproduction can keep
# the stable ploidy number. The changed ploidy will cause a fitness decrease (if the ploidy is not 45) by a Gaussian distribution
# (stablizing selection) and the sigma indicates the strength of selection.

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import math
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
class Populations(object):

    
    def __init__(self,nReps,nInds,nLoci,ploidy=45,mu=0.0001,selcoef=0.1, model_version ='ADD', sigma = 5):


        self.nReps = nReps
        self.nInds = nInds
        self.nLoci = nLoci
        
        self.soma = np.zeros((nReps,nInds,nLoci),dtype='int')
        self.germ = np.zeros((nReps,nInds,nLoci),dtype='int')

        self.ploidy = ploidy  # ploidy in each locus


        self.mu = mu
        self.selcoef = selcoef
        self.current_step = 0

        self.total_fitness = []
        self.total_log_fitness = []

        # self.equilibrium = False
        self.sex_gen = []

        self.model_version = model_version

        self.sigma = sigma

        # self.pre_gen_ploidy_matrix = np.ones((self.nReps,self.nInds,self.nLoci),dtype='int')*self.ploidy

        self.generate_ploidy_matrix()  # only for amitosis

        

    def generate_ploidy_matrix(self):

        initial_ploidy = np.ones((self.nReps,self.nInds,self.nLoci),dtype='int')*self.ploidy
        pre_change = np.random.multinomial (self.nLoci, [0.05, 0.90, 0.05], (self.nReps,self.nInds))

        change_matrix = []
        for i in range(self.nReps):
            t = []
            for j in range(self.nInds):
                y = [-1]*pre_change[i][j][0] + [0]*pre_change[i][j][1]+ [1]*pre_change[i][j][2]
                np.random.shuffle(y)
                t.append(y)
            change_matrix.append(t)

        change_matrix = np.array(change_matrix)

        self.ploidy_matrix = initial_ploidy + change_matrix


        

    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        ws = (self.soma.astype('float')/self.ploidy_matrix)*self.selcoef

        if self. model_version == 'ADD':
            fitnesses = 1 - np.sum(ws,axis=2)
            
        elif self. model_version == 'MUL':
            each_locus = (1-ws)*np.exp((-(self.ploidy_matrix -45)**2)/(2*(self.sigma**2)))
            # selection_effect = np.exp((-(self.ploidy_matrix -45)**2)/(2*(self.sigma**2) ))
            
            fitnesses = np.prod(each_locus, axis =2)
            
        fitnesses[fitnesses <= 0] = 0  

        return fitnesses


    def germ_fitness(self):
        """return a numpy array containing the germline fitness for each individual in each population"""
        gs = (self.germ.astype('float')/2)*self.selcoef

        if self. model_version == 'ADD':
            germ_fit = 1 - np.sum(gs,axis=2)
            
        elif self. model_version == 'MUL':
            each_germ_locus = 1-gs
            germ_fit = np.prod(each_germ_locus, axis =2)
            
        germ_fit[germ_fit <= 0] = 0  

        return germ_fit

               
            
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w,axis=1)
        totalw = np.expand_dims(total_w,axis=1)
        relfit = w/totalw
        return relfit

                
    def fixed_mutations(self):
        """return the number of fixed mutations withing each population"""
        nInds = self.soma.shape[1]
        fixed = np.sum(np.sum(self.soma==self.ploidy,axis=1)==nInds,axis=1)
        return fixed
            
   
    def all_wild_type_site(self):
        """return the number of fixed mutations withing each population"""
        nInds = self.soma.shape[1]
        all_wt_site = np.sum(np.sum(self.soma==0,axis=1)==nInds,axis=1)
        return all_wt_site


        
    #Wright_Fisher model
    def mutate_all_before(self):


        wt_soma = self.ploidy_matrix - self.soma   # how many wild type alleles left in each locus


        wt_germ = 2 - self.germ
        soma_mutations = np.random.binomial(wt_soma,self.mu)
        germ_mutations = np.random.binomial(wt_germ,self.mu)
        self.soma += soma_mutations
        self.germ += germ_mutations


        return
    
    
    
    def pick_parents_all(self):
        """Randomly choose N parents to produce offspring to populate the next generation.
        Each individual's probability of being chosen is weighted by its relative fitness.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        relfit = self.relative_fitness()
        csrelfit = np.cumsum(relfit,axis=1)
        randvals = np.random.random((nReps,nInds))
        parents = map(np.searchsorted,csrelfit,randvals)
        return parents
    
    
    
    def mitosis_all(self,parents):
        """Generate mitotic offspring from all individuals selected as parents.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)
        self.soma = self.soma[rReps,parents,]
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def amitosis_all(self,parents):
        """Generate amitotic offspring from all individuals selected as parents. Only one
        amitotic offspring is generated from each parent, so this method does not reflect
        the reciprocity of amitosis.
        Wright-Fisher model method."""
        nReps,nInds = self.soma.shape[0:2]        
        rReps = np.ones((nReps,nInds),dtype='int')*np.expand_dims(np.arange(nReps),axis=1)

        
        good = (self.ploidy_matrix[rReps, parents,]-self.soma[rReps,parents,])*2  # # potential problems here (self.ploidy_matrix may be smaller than self.soma)

        bad = self.soma[rReps,parents,]*2


        self.generate_ploidy_matrix()
        self.soma = np.random.hypergeometric(bad,good,self.ploidy_matrix)
        self.germ = self.germ[rReps,parents,]
        return
    
    
    def wright_fisher_step(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""
        # self.generate_ploidy_matrix()
        self.mutate_all_before()
        parents = self.pick_parents_all()
        if asex_type == 'amitosis':
            self.amitosis_all(parents)
        elif asex_type == 'mitosis':
            self.mitosis_all(parents)

        # self.pre_gen_ploidy_matrix = self.ploidy_matrix
        self.current_step += 1


        return
    
    
    

    
    
    #Asexul reproduction_WF model and Moran model
    
    def step(self,model='M',asex_type='amitosis',sex_freq=None):
        """Take populations through one time step if model='M', or one generation if model='WF'"""
        if model == 'M':
            self.moran_step(asex_type)
        elif model == 'WF':
            self.wright_fisher_step(asex_type)
        return 
    


    def make_gametes(self, parents):


        rReps = np.ones((self.nReps,self.nInds),dtype='int')*np.expand_dims(np.arange(self.nReps),axis=1)
        gametes = np.random.hypergeometric(self.germ[rReps,parents,],2-self.germ[rReps,parents,],1)

        return gametes




    def make_zygotes(self, random_mating=False):

        # self.ploidy_matrix = np.ones((self.nReps,self.nInds,self.nLoci),dtype='int')*self.ploidy


        if random_mating == False:
            parents = self.pick_parents_all()

            these_gametes = self.make_gametes(parents)
            those_gametes = self.make_gametes(parents)

        else: 
            first_parents = self.pick_parents_all()
            second_parents = self.pick_parents_all()

            these_gametes = self.make_gametes(first_parents)
            those_gametes = self.make_gametes(second_parents)           


        zygotes = these_gametes + those_gametes

        return zygotes




    def sex(self,random_mating=False):

        self.mutate_all_before()

        zygotes = self.make_zygotes(random_mating)  
        self.germ = zygotes  
        
        hetcount = len(self.germ[self.germ==1])  
        hetvals = np.random.binomial(self.ploidy,0.5,hetcount) 

        self.soma[self.germ==1] = hetvals 
        self.soma[self.germ==2] = self.ploidy 
        self.soma[self.germ==0] = 0  

        self.ploidy_matrix = np.ones((self.nReps,self.nInds,self.nLoci),dtype='int')*self.ploidy
        self.current_step +=1

        return 



    
    def get_results(self):
        """calculate stuff like mean fitness, Gst, and number of fixed mutations"""


        # fitness
        W = self.fitness()
        log_W = np.log(W)

        mW = np.nanmean(W)
        log_mW = np.log(mW)
        sdW = np.nanstd(W)
        semW = stats.sem(W,None)

        mlog_W = np.nanmean(log_W)
        sdlog_W = np.nanstd(log_W)
        semlog_W = stats.sem(log_W,None)

        total_log_var = []
        total_fit_var = []
        for i in range(self.nReps):
            log_var = np.var(log_W[i])
            fit_var = np.var(W[i])

            total_log_var.append(log_var)
            total_fit_var.append(fit_var)

        varlog_W = np.nanmean(total_log_var)
        varW = np.nanmean(total_fit_var)
 

        # germ fitness
        GF = self.germ_fitness()
        log_GF = np.log(GF)

        mGF = np.nanmean(GF)
        log_mGF = np.log(mGF)
        sdGF = np.nanstd(GF)
        semGF = stats.sem(GF,None)

        mlog_GF = np.nanmean(log_GF)
        sdlog_GF = np.nanstd(log_GF)
        semlog_GF = stats.sem(log_W,None)

        total_log_var_g = []
        total_fit_var_g = []
        for i in range(self.nReps):
            log_var_g = np.var(log_GF[i])
            fit_var_g = np.var(GF[i])

            total_log_var_g.append(log_var_g)
            total_fit_var_g.append(fit_var_g)

        varlog_GF = np.nanmean(total_log_var_g)
        varGF = np.nanmean(total_fit_var_g)     


        # ploidy change
        PN = self.ploidy_matrix

        mPN = np.nanmean(PN)
        sdPN = np.nanstd(PN)
        semPN = stats.sem(sdPN, None)

        total_ploidy_var = []
        for i in range(self.nReps):
            ploidy_var = np.var(PN[i])

            total_ploidy_var.append(ploidy_var)

        varPN = np.nanmean(total_ploidy_var)



        F = self.fixed_mutations()
        mF = np.nanmean(F)
        sdF = np.nanstd(F)
        semF = stats.sem(F,None)

       
        # return [mW,log_mW, sdW,semW,mlog_W, sdlog_W, semlog_W, varlog_W, varW]
        return [mW,log_mW, sdW,semW,mlog_W, sdlog_W, semlog_W, varlog_W, varW, mGF,log_mGF, sdGF,semGF,mlog_GF, sdlog_GF, semlog_GF, varlog_GF, varGF,\
        mPN, sdPN, semPN, varPN, mF,sdF,semF]        
    
    

    def monitor_population_mutation(self):
        '''Monitor the mutation dynamics in population level'''

        fixed = self.fixed_mutations()
        all_wt_site = self.all_wild_type_site()
        poly_site = self.nLoci-fixed-all_wt_site


        overall_mu_freq = np.sum(self.soma, axis =1)/(self.ploidy*self.nInds)

        value_all_poly_freq_pl = []
        value_all_poly_var_pl = []


        for i in range(self.nReps):
            pop_poly_freq_pl = []
            # pop_poly_var_pl = []

            for j in range(self.nLoci):
                if overall_mu_freq[i][j]!= 0 and overall_mu_freq[i][j]!= 1:
                    pop_poly_freq_pl.append(overall_mu_freq[i][j])

            if len(pop_poly_freq_pl) == 0:
                value_pop_poly_freq = 0
                value_pop_poly_var = 0
            elif len(pop_poly_freq_pl) == 1:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = 0
            else:
                value_pop_poly_freq = np.nanmean(pop_poly_freq_pl)
                value_pop_poly_var = np.var(pop_poly_freq_pl)


            value_all_poly_freq_pl.append(value_pop_poly_freq)
            value_all_poly_var_pl.append(value_pop_poly_var)


        meanPopFM = np.nanmean(fixed)
        stdPopFM = np.nanstd(fixed)
        sePopFM = stdPopFM/(len(fixed)**0.5)
        semPopFM = stats.sem(fixed,None)

        meanPopPM = np.nanmean(poly_site)
        stdPopPM = np.nanstd(poly_site)
        sePopPM = stdPopPM/(len(poly_site)**0.5)
        semPopPM = stats.sem(poly_site,None)     

        meanPopPF = np.nanmean(value_all_poly_freq_pl)   
        stdPopPF = np.nanstd(value_all_poly_freq_pl)
        sePopPF = stdPopPF/(len(value_all_poly_freq_pl)**0.5)
        semPopPF =  stats.sem(value_all_poly_freq_pl,None)

        meanPopPV = np.nanmean(value_all_poly_var_pl)
        stdPopPV = np.nanstd(value_all_poly_var_pl)
        sePopPV = stdPopPV/(len(value_all_poly_var_pl)**0.5)
        semPopPV = stats.sem(value_all_poly_var_pl,None)

        return [meanPopFM, stdPopFM, sePopFM, semPopFM, meanPopPM, stdPopPM, sePopPM, semPopPM, \
        meanPopPF, stdPopPF, sePopPF, semPopPF, meanPopPV, stdPopPV, sePopPV, semPopPV]



    
    def monitor_individual_mutation(self):
        '''Monitor the mutation dynamics in each site in each individual in each population'''
        # nReps, nInds, nLoci = self.soma.shape[0:3]
        # ploidy = self.ploidy
        
        all_fix_site = []
        all_poly_site = []
        
        all_poly_freq = []
        all_poly_var = []
        
        for i in range(self.nReps):
            pop_fix_site = []
            pop_poly_site = []
            
            pop_poly_freq = []
            pop_poly_var = []
            
            for j in range(self.nInds):
                ind_fix_site = 0
                ind_poly_site = 0
            
                ind_poly_freq = []
#                 ind_poly_var = 0
                
                for k in range(self.nLoci): # loci level
                    if self.soma[i][j][k] == self.ploidy:
                        ind_fix_site +=1
                    elif self.soma[i][j][k]>0 and self.soma[i][j][k]<self.ploidy:
                        ind_poly_site +=1
                        ind_poly_freq.append(self.soma[i][j][k]/self.ploidy)  # the freq in each poly site
                        
                ind_poly_freq_mean = np.nanmean(ind_poly_freq) 
                # print '1', ind_poly_freq_mean
                        
                if len(ind_poly_freq) ==0 or len(ind_poly_freq) ==1: 
                    ind_poly_var = 0
                else:
                    ind_poly_var = np.var(ind_poly_freq) # this is the variance of each individual
                    
                
                pop_fix_site.append(ind_fix_site) # no. of fixed mutation site in each individual in a population
                pop_poly_site.append(ind_poly_site) # no. of poly mutation site in each individual in a population
                
                pop_poly_freq.append(ind_poly_freq_mean) # the poly freq of each individual in a population
                pop_poly_var.append(ind_poly_var) # the variance of each individual in a population
                
            
            all_fix_site.append(np.nanmean(pop_fix_site)) # mean no. of fixed mutation site in a population
            all_poly_site.append(np.nanmean(pop_poly_site)) # mean no. of ploy mutation site in a population
            all_poly_freq.append(np.nanmean(pop_poly_freq)) # mean poly freq in a population
            all_poly_var.append(np.nanmean(pop_poly_var)) # mean poly variance in a population
            
            
        mean_fix_site = np.nanmean(all_fix_site)
        std_fix_site = np.nanstd(all_fix_site)
        se_fix_site = std_fix_site/(len(all_fix_site)**0.5)
        sem_fix_site = stats.sem(all_fix_site,None)

        mean_poly_site = np.nanmean(all_poly_site)
        std_poly_site = np.nanstd(all_poly_site)
        se_poly_site = std_poly_site/(len(all_poly_site)**0.5)
        sem_poly_site = stats.sem(all_poly_site,None)

        mean_poly_freq = np.nanmean(all_poly_freq)
        std_poly_freq = np.nanstd(all_poly_freq)
        se_poly_freq = std_poly_freq/(len(all_poly_freq)**0.5)
        sem_poly_freq = stats.sem(all_poly_freq,None)

        mean_poly_var = np.nanmean(all_poly_var)
        std_poly_var = np.nanstd(all_poly_var)
        se_poly_var = std_poly_var/(len(all_poly_var)**0.5)
        sem_poly_var = stats.sem(all_poly_var,None)
        

        return [mean_fix_site, std_fix_site, se_fix_site, sem_fix_site, mean_poly_site, std_poly_site, se_poly_site, sem_poly_site, \
        mean_poly_freq, std_poly_freq, se_poly_freq, sem_poly_freq, mean_poly_var, std_poly_var, se_poly_var, sem_poly_var]


        
        
    def simulate(self,stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
 
        results = [self.get_results()]

        ind_mutations = [self.monitor_individual_mutation()]
        pop_mutations = [self.monitor_population_mutation()]


        start = time.time()
    
        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
                self.sex_gen.append(self.current_step)
               
            else:
                self.step(model,asex_type,sex_freq)

            
            if self.current_step%strides == 0:
                results.append(self.get_results())
                ind_mutations.append(self.monitor_individual_mutation())
                pop_mutations.append(self.monitor_population_mutation())

                
        # colnames = ['meanFit','log_meanFit','sdFit','semFit','mean_logFit','sdlog_Fit', 'semlog_Fit', 'varlog_Fit', 'varFit']
        colnames = ['meanFit','log_meanFit','sdFit','semFit','mean_logFit','sdlog_Fit', 'semlog_Fit', 'varlog_Fit', 'varFit', \
        'meanFit_Germ','log_meanFit_Germ','sdFit_Germ','semFit_Germ','mean_logFit_Germ','sdlog_Fit_Germ', 'semlog_Fit_Germ', 'varlog_Fit_Germ', 'varFit_Germ',\
        'meanPloidy','sdPloidy','semPloidy', 'VarPloidy', 'meanFM','sdFM','semFM']

        ind_mutation_colnames = ['meanFM', 'stdFM', 'seFM', 'semFM', 'meanPM', 'stdPM', 'sePM', 'semPM', \
        'meanPF','stdPF', 'sePF', 'semPF', 'meanPV', 'stdPV', 'sePV', 'semPV']
        pop_mutation_colnames = ['meanPopFM', 'stdPopFM', 'sePopFM', 'semPopFM', 'meanPopPM', 'stdPopPM', 'sePopPM', 'semPopPM',\
        'meanPopPF', 'stdPopPF', 'sePopPF', 'semPopPF', 'meanPopPV', 'stdPopPV', 'sePopPV', 'semPopPV']

        
        results = pd.DataFrame(np.array(results),columns=colnames)

        ind_mutations = pd.DataFrame(np.array(ind_mutations),columns=ind_mutation_colnames)
        pop_mutations = pd.DataFrame(np.array(pop_mutations),columns=pop_mutation_colnames)

        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results, ind_mutations, pop_mutations
        


    def simulateNsave(self,outfile_1,outfile_2, outfile_3, stepcount,model='M',asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):
        """same as simulate method except results are written to the file specified by outfile"""
        results = self.simulate(stepcount,model,asex_type,sex_freq,random_mating,strides)
        
        result = results[0]
        result.to_csv(outfile_1,index=False)
        
        ind_mutation = results[1]
        ind_mutation.to_csv(outfile_2, index=False)

        pop_mutation = results[2]
        pop_mutation.to_csv(outfile_3, index=False)
        
        return



def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)     
         